# Data Analysis for READ-MAS Agents

In this notebook, the experiment data is loaded, prepared, explored and statistically analyzed.


## Import Libraries and Define Constants

In [179]:
import numpy as np
import pandas as pd
import  pingouin 
import matplotlib.pyplot as plt
import seaborn as sns

# The parent path to the benchmark experiment runs
EXPERIMENT_PATH = '../benchmark_runs/'

# Column renames for LLM as a Judge benchmark runs
RENAME_BENCHMARK = {
  'runs/benchmark_runs/metrics.json:agent':                    'agent',
  'runs/benchmark_runs/metrics.json:model':                    'model',
  'runs/benchmark_runs/metrics.json:rag':                      'rag',
  'runs/benchmark_runs/metrics.json:DesignAccuracy [GEval]':   'design_accuracy',
  'DesignAccuracy [GEval]':                                    'design_accuracy',
  'runs/benchmark_runs/metrics.json:faithfulness (ragas)':     'ragas_faithfulness',
  'runs/benchmark_runs/metrics.json:RAGAS':                    'ragas',
}
# Columns to keep in the benchmark runs data frame
KEEP_BENCHMARK = ['agent', 'model', 'rag', 'design_accuracy', 'ragas_faithfulness', 'ragas']

# Column renames for code benchmark runs
RENAME_CODE_BENCHMARK = {
  'code.agent_name':                                          'agent',
  'code.model':                                               'model',
  'code.rag':                                                 'rag',
  'pass@1':                                                   'pass_at_1',
  'pass@1plus':                                               'pass_at_1plus',
  'code.dataset':                                             'benchmark',
}

# Columns to keep for code benchmark runs
KEEP_CODE_BENCHMARK = ['agent', 'model', 'rag', 'pass_at_1', 'pass_at_1plus', 'benchmark']

# Column renames for plain LLM code benchmark runs
RENAME_LLM_BENCHMARK = {
  'runs/llm_benchmark_runs/metrics.json:model':                     'model',
  'runs/llm_benchmark_runs/metrics.json:pass@1':                    'pass_at_1',
  'runs/llm_benchmark_runs/metrics.json:pass@1plus':                'pass_at_1plus',
  'runs/llm_benchmark_runs/metrics.json:benchmark':                 'benchmark',
}

# Columns to keep for plain LLM code benchmark runs
KEEP_LLM_BENCHMARK = ['agent', 'model', 'pass_at_1', 'pass_at_1plus', 'benchmark']

# LLM as a Judge and RAGAS metrics
BENCHMARK_METRICS = ['design_accuracy', 'ragas_faithfulness', 'ragas']

# HumanEval and MBPP metrics for code benchmarking
CODE_METRICS = ['pass_at_1', 'pass_at_1plus']

# Metrics without RAGAS for RQ5
NO_RAGAS_METRICS = ['design_accuracy']

# Constants for experiment groups corresponding to the Research Questions
# Mapping of custom benchmarks and metrics
BENCHMARK_COL_METRICS_MAP = {
  'benchmark': {
    'rename_cols': RENAME_BENCHMARK,
    'keep_cols': KEEP_BENCHMARK,
    'metrics': BENCHMARK_METRICS
  }
}

# Mapping of custom benchmarks and metrics for NO RAGAS use case
NO_RAGAS_BENCHMARK_COL_METRICS_MAP = {
  'benchmark': {
    'rename_cols': RENAME_BENCHMARK,
    'keep_cols': KEEP_BENCHMARK,
    'metrics': NO_RAGAS_METRICS
  }
}

# Mapping of coding benchmarks and metrics
CODE_BENCHMARK_COL_METRICS_MAP = {
  'humaneval': {
    'rename_cols': RENAME_CODE_BENCHMARK,
    'keep_cols': KEEP_CODE_BENCHMARK,
    'metrics': CODE_METRICS    
  },
  'mbpp': {
    'rename_cols': RENAME_CODE_BENCHMARK,
    'keep_cols': KEEP_CODE_BENCHMARK,
    'metrics': CODE_METRICS    
  }
}

# Code benchmark mappings with no RAG for LLM vs agent comparison (RQ4)
CODE_LLM_COL_METRICS_MAP = {
  'humaneval': {
    'rename_cols': RENAME_CODE_BENCHMARK | RENAME_LLM_BENCHMARK,
    'keep_cols': KEEP_LLM_BENCHMARK,
    'metrics': CODE_METRICS    
  },
  'mbpp': {
    'rename_cols': RENAME_CODE_BENCHMARK | RENAME_LLM_BENCHMARK,
    'keep_cols': KEEP_LLM_BENCHMARK,
    'metrics': CODE_METRICS    
  }
}

# RQ1: Custom benchmark, HumanEval and MBPP code benchmark experiment datasets
RQ1_BENCHMARK_FILES = {
    'benchmark': ('single_agent_benchmark_gemini.csv', 'read_agent_benchmark_gemini.csv'),
    'humaneval': ('single_agent_code_benchmark_humaneval_gemini.csv',
                'read_agent_code_benchmark_humaneval_gemini.csv'),
    'mbpp':      ('single_agent_code_benchmark_mbpp_gemini.csv',
                'read_agent_code_benchmark_mbpp_gemini.csv'),
}

# Columns and metrics map combining custom and code benchmarks.
FULL_COL_METRICS_MAP = BENCHMARK_COL_METRICS_MAP | CODE_BENCHMARK_COL_METRICS_MAP

# No RAGAS columns and metrics map combining custom and code benchmarks.
NO_RAGAS_COL_METRICS_MAP = NO_RAGAS_BENCHMARK_COL_METRICS_MAP | CODE_BENCHMARK_COL_METRICS_MAP

# RQ2: Custom benchmark, HumanEval and MBPP code benchmark experiment datasets
RQ2_BENCHMARK_FILES = {
    'benchmark': ('read_agent_benchmark_claude.csv', 'read_agent_benchmark_gemini.csv', 'read_agent_benchmark_openai.csv'),
    'humaneval': ('read_agent_code_benchmark_humaneval_claude.csv',
                'read_agent_code_benchmark_humaneval_gemini.csv', 
                'read_agent_code_benchmark_humaneval_openai.csv'),
    'mbpp':      ('single_agent_code_benchmark_mbpp_gemini.csv',
                'read_agent_code_benchmark_mbpp_gemini.csv'),
}

# RQ4: Custom benchmark, HumanEval and MBPP code benchmark experiment datasets
RQ4_BENCHMARK_FILES = {
    'humaneval': ('read_agent_code_benchmark_humaneval_claude.csv',
                'llm_code_benchmark_humaneval_claude.csv'),
    #'mbpp':      ('read_agent_code_benchmark_mbpp_claude.csv',
    #            'llm_code_benchmark_mbpp_claude.csv'),
}

# RQ5: Custom benchmark, HumanEval and MBPP code benchmark experiment datasets
RQ5_BENCHMARK_FILES = {
    'benchmark': ('read_agent_benchmark_gemini.csv', 'read_agent_benchmark_gemini_no_rag.csv'),
    #'humaneval': ('read_agent_code_benchmark_humaneval_gemini.csv',
    #            'read_agent_code_benchmark_humaneval_gemini_no_rag.csv'),
    #'mbpp':      ('read_agent_code_benchmark_mbpp_gemini.csv',
    #            'read_agent_code_benchmark_mbpp_gemini_no_rag.csv'),
}

## Helper Functions for Experiment Data Loading and Perform ANOVA

Helper functions

In [161]:
def load_experiment_dataset(filename: str, rename_cols: dict[str, str], keep_cols: list[str]):
    """This function loads an experiment data in CSV format into a data frame, filters out the baselines, renames the columns, and keeps only the columns needed for the analysis.
    
    Args:
        filename: the relative path to the experiment dataset
        rename_cols: a dict to rename the dataset columns
        keep_cols: the columns to keep in the data frame
        
    Returns:
        A Pandas data frame
    """
    df = pd.read_csv(EXPERIMENT_PATH + filename)
    df = df[df['typ'].isin(['branch_base', 'branch_commit'])]
    df = df.rename(columns=rename_cols)
    
     # Set agent to llm for LLM benchmark datasets
    if 'agent' not in df.columns:
      df['agent'] = 'llm'

    return df[keep_cols]


In [ ]:
def get_group_dfs(group_files: dict[str, tuple[str, ...]], group_col_metrics_map: dict[str, dict[str, str]]):
  group_dfs = []
  for benchmark, files in group_files.items():
    config = group_col_metrics_map[benchmark]
    rename_cols = config['rename_cols']
    keep_cols = config['keep_cols']
    metrics = config['metrics']
    
    df_list = [
        load_experiment_dataset(EXPERIMENT_PATH + f, rename_cols, keep_cols) 
        for f in files
    ]
    
    group_df = pd.concat(df_list, ignore_index=True)
        
    group_dfs.append({'benchmark': benchmark, 'group_df': group_df, 'metrics': metrics})

  return group_dfs

In [170]:
def perform_anova(benchmark, group_df, metrics, group):
    """This function performs ANOVA assumption checks, ANOVA, and post hoc tests 
    and displays results as Markdown tables for dissertation use.
    
    Args:
        benchmark: the benchmark for which the ANOVA is performed
        group_df: the combined data frame for a particular hypothesis test
        metrics: the list of metrics serving as a target for the ANOVA tests
        group: the categorical variable ANOVA compares
    """
    
    # Explicit coercion since LLM file stores pass values as strings, which causes errors in pingouin.
    group_df = group_df.copy()
    for metric in metrics:
        group_df[metric] = pd.to_numeric(group_df[metric], errors='coerce')
    
    print(f"\n# Benchmark: {benchmark}")
    
    for metric in metrics:
        print(f"\n## Metric: {metric}")
        
        # ANOVA Assumption checks
        norm = pingouin.normality(data=group_df, dv=metric, group=group, method='shapiro')
        print("\n#### Shapiro-Wilk Normality Test")
        print(f"*p > 0.05 indicates normality*")
        print(norm[['W', 'pval', 'normal']].to_markdown(index=False))
        
        eq_var = pingouin.homoscedasticity(data=group_df, dv=metric, group=group, method='levene')
        print("\n#### Levene Homoscedasticity Test")
        print(f"*p > 0.05 indicates equal variances*")
        print(eq_var[['W', 'pval', 'equal_var']].to_markdown(index=False))
        
        normal = norm['normal'].all()
        equal_var = eq_var['equal_var'].iloc[0]
        
        # ANOVA/Kruskal-Wallis tests best on assumption check results
        if not normal:
            kw = pingouin.kruskal(data=group_df, dv=metric, between=group)
            print(f"\n#### Kruskal-Wallis Test (Non-Parametric)")
            print(kw.to_markdown(index=False))
            
            # Convert series to dataframe for markdown formatting
            medians = group_df.groupby(group)[metric].median().reset_index()
            medians.columns = [group, 'Median']
            print(f"\n#### Group Medians")
            print(medians.to_markdown(index=False))
            
        elif not equal_var:
            aov = pingouin.welch_anova(data=group_df, dv=metric, between=group)
            print(f"\n#### Welch's ANOVA")
            print(aov.to_markdown(index=False))
            print(f"\n**Summary:** F({aov.iloc[0]['ddof1']:.0f}, {aov.iloc[0]['ddof2']:.2f}) = {aov.iloc[0]['F']:.4f}, p = {aov.iloc[0]['p_unc']:.4f}")
            
            gh = pingouin.pairwise_gameshowell(data=group_df, dv=metric, between=group)
            print(f"\n#### Games-Howell Post Hoc Test")
            print(gh.to_markdown(index=False))
            
        else:
            aov = pingouin.anova(data=group_df, dv=metric, between=group, detailed=True)
            print(f"\n#### One-way ANOVA")
            print(aov.to_markdown(index=False))
            print(f"\n**Summary:** F({aov.iloc[0]['DF']}, {aov.iloc[1]['DF']}) = {aov.iloc[0]['F']:.4f}, p = {aov.iloc[0]['p_unc']:.4f}")
            
            tukey = pingouin.pairwise_tukey(data=group_df, dv=metric, between=group)
            print(f"\n#### Tukey HSD Post Hoc Test")
            print(tukey.to_markdown(index=False))

## Load Experiment Datasets

Load the experiment datasets and filter the experiments.

In [180]:
# RQ1 grouped datasets
rq1_group_dfs = get_group_dfs(RQ1_BENCHMARK_FILES, FULL_COL_METRICS_MAP)

for group in rq1_group_dfs:
  print(f"Benchmark {group['benchmark']}  Metrics {group['metrics']}\n")
  print(f"RQ1 combined group dataset for {group['benchmark']}: \n")
  print(group['group_df'].head())
  print("\n\n")

# RQ2 grouped datasets
rq2_group_dfs = get_group_dfs(RQ2_BENCHMARK_FILES, FULL_COL_METRICS_MAP)

for group in rq2_group_dfs:
  print(f"RQ2 combined group dataset for {group['benchmark']}: \n")
  print(group['group_df'].head())
  print("\n\n")
  
# RQ4 grouped datasets
rq4_group_dfs = get_group_dfs(RQ4_BENCHMARK_FILES, CODE_LLM_COL_METRICS_MAP)

for group in rq4_group_dfs:
  print(f"RQ4 combined group dataset for {group['benchmark']}: \n")
  print(group['group_df'].head())
  print("\n\n")
  
# RQ5 grouped datasets
rq5_group_dfs = get_group_dfs(RQ5_BENCHMARK_FILES, NO_RAGAS_COL_METRICS_MAP)

for group in rq5_group_dfs:
  print(f"RQ5 combined group dataset for {group['benchmark']}: \n")
  print(group['group_df'].head())
  print("\n\n")


Benchmark benchmark  Metrics ['design_accuracy', 'ragas_faithfulness', 'ragas']

RQ1 combined group dataset for benchmark: 

          agent             model  ...  ragas_faithfulness  ragas
0  single_agent  gemini-2.5-flash  ...               0.910  0.697
1  single_agent  gemini-2.5-flash  ...               0.703  0.652
2  single_agent  gemini-2.5-flash  ...               0.846  0.670
3  single_agent  gemini-2.5-flash  ...               0.894  0.675
4  single_agent  gemini-2.5-flash  ...               0.816  0.639

[5 rows x 6 columns]



Benchmark humaneval  Metrics ['pass_at_1', 'pass_at_1plus']

RQ1 combined group dataset for humaneval: 

          agent             model   rag  pass_at_1  pass_at_1plus  benchmark
0  single_agent  gemini-2.5-flash  True      0.918          0.866  humaneval
1  single_agent  gemini-2.5-flash  True      0.927          0.863  humaneval
2  single_agent  gemini-2.5-flash  True      0.933          0.887  humaneval
3  single_agent  gemini-2.5-flash  True  

## RQ1: Compare Single and Multi Agents

In [172]:
for group in rq1_group_dfs:
  perform_anova(group['benchmark'], group['group_df'], group['metrics'], 'agent')


# Benchmark: benchmark

## Metric: design_accuracy

#### Shapiro-Wilk Normality Test
*p > 0.05 indicates normality*
|        W |     pval | normal   |
|---------:|---------:|:---------|
| 0.918068 | 0.341123 | True     |
| 0.947691 | 0.64127  | True     |

#### Levene Homoscedasticity Test
*p > 0.05 indicates equal variances*
|      W |       pval | equal_var   |
|-------:|-----------:|:------------|
| 9.7183 | 0.00594821 | False       |

#### Welch's ANOVA
| Source   |   ddof1 |   ddof2 |       F |     p_unc |      np2 |
|:---------|--------:|--------:|--------:|----------:|---------:|
| agent    |       1 |  13.256 | 6.41633 | 0.0246842 | 0.262789 |

**Summary:** F(1, 13.26) = 6.4163, p = 0.0247

#### Games-Howell Post Hoc Test
| A          | B            |   mean_A |   mean_B |   diff |        se |       T |     df |      pval |   hedges |
|:-----------|:-------------|---------:|---------:|-------:|----------:|--------:|-------:|----------:|---------:|
| read_agent | single_agent |

## RQ2: Multi-agent performance among LLMs

In [173]:
for group in rq2_group_dfs:
  perform_anova(group['benchmark'], group['group_df'], group['metrics'], 'model')



# Benchmark: benchmark

## Metric: design_accuracy

#### Shapiro-Wilk Normality Test
*p > 0.05 indicates normality*
|        W |     pval | normal   |
|---------:|---------:|:---------|
| 0.974542 | 0.929467 | True     |
| 0.947691 | 0.64127  | True     |
| 0.924307 | 0.394278 | True     |

#### Levene Homoscedasticity Test
*p > 0.05 indicates equal variances*
|       W |      pval | equal_var   |
|--------:|----------:|:------------|
| 4.60499 | 0.0190208 | False       |

#### Welch's ANOVA
| Source   |   ddof1 |   ddof2 |       F |      p_unc |      np2 |
|:---------|--------:|--------:|--------:|-----------:|---------:|
| model    |       2 | 15.6912 | 7.84425 | 0.00435042 | 0.475008 |

**Summary:** F(2, 15.69) = 7.8443, p = 0.0044

#### Games-Howell Post Hoc Test
| A                           | B                 |   mean_A |   mean_B |   diff |         se |        T |      df |       pval |   hedges |
|:----------------------------|:------------------|---------:|---------:|-------

AssertionError: Data must have at least two columns.

## RQ3: Compare READ-MAS with Baseline Frameworks

## RQ4: Compare READ-MAS with LLM

In [174]:
for group in rq4_group_dfs:
  perform_anova(group['benchmark'], group['group_df'], group['metrics'], 'agent')


# Benchmark: humaneval

## Metric: pass_at_1

#### Shapiro-Wilk Normality Test
*p > 0.05 indicates normality*
|        W |     pval | normal   |
|---------:|---------:|:---------|
| 0.912598 | 0.299321 | True     |
| 0.870884 | 0.102365 | True     |

#### Levene Homoscedasticity Test
*p > 0.05 indicates equal variances*
|        W |     pval | equal_var   |
|---------:|---------:|:------------|
| 0.152381 | 0.700851 | True        |

#### One-way ANOVA
| Source   |        SS |   DF |        MS |       F |       p_unc |        np2 |
|:---------|----------:|-----:|----------:|--------:|------------:|-----------:|
| agent    | 0.0114242 |    1 | 0.0114242 | 140.174 |   6.266e-10 |   0.886201 |
| Within   | 0.001467  |   18 | 8.15e-05  | nan     | nan         | nan        |

**Summary:** F(1, 18) = 140.1742, p = 0.0000

#### Tukey HSD Post Hoc Test
| A   | B          |   mean_A |   mean_B |    diff |         se |        T |     p_tukey |   hedges |
|:----|:-----------|---------:|---------:

## RQ5: Compare READ-MAS with and without RAG

In [181]:
for group in rq5_group_dfs:
  perform_anova(group['benchmark'], group['group_df'], group['metrics'], 'rag')


# Benchmark: benchmark

## Metric: design_accuracy

#### Shapiro-Wilk Normality Test
*p > 0.05 indicates normality*
|        W |     pval | normal   |
|---------:|---------:|:---------|
| 0.947691 | 0.64127  | True     |
| 0.891808 | 0.177702 | True     |

#### Levene Homoscedasticity Test
*p > 0.05 indicates equal variances*
|        W |     pval | equal_var   |
|---------:|---------:|:------------|
| 0.144406 | 0.708386 | True        |

#### One-way ANOVA
| Source   |       SS |   DF |       MS |         F |      p_unc |        np2 |
|:---------|---------:|-----:|---------:|----------:|-----------:|-----------:|
| rag      | 0.00128  |    1 | 0.00128  |   3.40426 |   0.081547 |   0.159046 |
| Within   | 0.006768 |   18 | 0.000376 | nan       | nan        | nan        |

**Summary:** F(1, 18) = 3.4043, p = 0.0815

#### Tukey HSD Post Hoc Test
| A     | B    |   mean_A |   mean_B |   diff |         se |       T |   p_tukey |   hedges |
|:------|:-----|---------:|---------:|-------:|--